<div dir="rtl" align="right">

# پروژه: PCA و طبقه‌بند softmax فقط با NumPy

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## هدف
در این پروژه همه چیزهایی را که یاد گرفتی کنار هم می‌گذاری و **بدون scikit-learn** دو الگوریتم واقعی می‌سازی. داده: گل‌های زنبق (150 نمونه، 4 ویژگی، 3 کلاس).

| بخش | ریاضی‌ای که استفاده می‌شود | درس مربوط |
|---|---|---|
| استانداردسازی | میانگین، انحراف معیار، نمره z | آمار توصیفی |
| PCA | ماتریس کوواریانس، بردار ویژه، SVD | مقدار ویژه، SVD |
| طبقه‌بند | ضرب ماتریسی، softmax | ماتریس‌ها، ریاضی طبقه‌بندی |
| تابع خطا | آنتروپی متقاطع | درست‌نمایی، آنتروپی |
| آموزش | گرادیان، گرادیان کاهشی | مشتق جزئی، گرادیان کاهشی |
| بررسی | مشتق عددی | مشتق |

## بخش 1: PCA در چهار گام
| گام | فرمول |
|---|---|
| 1. استانداردسازی | $Z=(X-\mu)/\sigma$ |
| 2. ماتریس کوواریانس | $C=\dfrac{1}{n-1}Z^\top Z$ |
| 3. تجزیه ویژه | $C=V\Lambda V^\top$ و مرتب‌سازی نزولی |
| 4. تصویر کردن | $T=ZV_k$ |

سهم هر مؤلفه از واریانس: $\lambda_i/\sum_j\lambda_j$.

> [چپ: چهار ویژگی گل زنبق که به دو مؤلفه اصلی تصویر شده‌اند. راست: سهم هر مؤلفه از واریانس.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_pca_iris.png)

*چپ: چهار ویژگی گل زنبق که به دو مؤلفه اصلی تصویر شده‌اند. راست: سهم هر مؤلفه از واریانس.*



</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris          # used only to load the data

iris = load_iris(); X, y = iris.data, iris.target
Z = (X - X.mean(axis=0)) / X.std(axis=0)                 # 1. standardize
C = Z.T @ Z / (len(Z) - 1)                               # 2. covariance matrix
vals, vecs = np.linalg.eigh(C)                           # 3. eigendecomposition (ascending)
order = np.argsort(vals)[::-1]; vals, vecs = vals[order], vecs[:, order]
T = Z @ vecs[:, :2]                                      # 4. project on 2 components

print("eigenvalues:", vals.round(3))
print("variance explained:", (vals / vals.sum()).round(3), " first two:", round(vals[:2].sum() / vals.sum(), 3))
print("first component:", vecs[:, 0].round(3))

# The SVD gives the same eigenvalues
s = np.linalg.svd(Z, compute_uv=False)
print("from the SVD:", (s ** 2 / (len(Z) - 1)).round(3))

for k, name in enumerate(iris.target_names):
    plt.scatter(T[y == k, 0], T[y == k, 1], s=15, label=name)
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.legend(); plt.title("PCA from scratch"); plt.show()

<div dir="rtl" align="right">



دو مؤلفه اول حدود 96٪ واریانس را نگه می‌دارند. SVD همان جواب را می‌دهد: $\lambda_i=\sigma_i^2/(n-1)$.

## بخش 2: رگرسیون softmax
مدل برای n نمونه، d ویژگی و K کلاس:

$$Z=XW+b\qquad P=\mathrm{softmax}(Z)\qquad L=-\frac1n\sum_i\ln P_{i,y_i}$$

گرادیان‌ها (از درس ریاضی طبقه‌بندی، به شکل ماتریسی):

$$\frac{\partial L}{\partial W}=\frac1nX^\top(P-Y)\qquad\frac{\partial L}{\partial b}=\frac1n\sum_i(P_i-Y_i)$$

که Y ماتریس one-hot برچسب‌هاست.

| نماد | شکل | در این پروژه |
|---|---|---|
| X | $n\times d$ | $120\times4$ |
| W | $d\times K$ | $4\times3$ |
| b | K | 3 |
| P و Y | $n\times K$ | $120\times3$ |



</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris(); X, y = iris.data, iris.target
rng = np.random.default_rng(0)
idx = rng.permutation(150); tr, te = idx[:120], idx[120:]
mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)           # statistics from the training set only
Xtr, Xte = (X[tr] - mu) / sd, (X[te] - mu) / sd
Y = np.eye(3)[y[tr]]                                     # one-hot labels, 120 x 3

def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

def loss_and_grads(W, b):
    P = softmax(Xtr @ W + b)
    loss = -np.mean(np.log(P[np.arange(120), y[tr]]))
    return loss, Xtr.T @ (P - Y) / 120, (P - Y).mean(axis=0)

W = np.zeros((4, 3)); b = np.zeros(3)
print("initial loss:", round(loss_and_grads(W, b)[0], 4), "  ln 3 =", round(np.log(3), 4))

# Gradient check on one weight, at a random point
Wr = rng.normal(0, 0.5, (4, 3)); eps = 1e-6
Wp = Wr.copy(); Wp[1, 2] += eps; Wm = Wr.copy(); Wm[1, 2] -= eps
numeric = (loss_and_grads(Wp, b)[0] - loss_and_grads(Wm, b)[0]) / (2 * eps)
print("gradient check:", round(numeric, 6), "vs", round(loss_and_grads(Wr, b)[1][1, 2], 6))

for epoch in range(1, 501):
    loss, gW, gb = loss_and_grads(W, b)
    W -= 0.5 * gW; b -= 0.5 * gb
    if epoch in (1, 10, 100, 500): print(f"epoch {epoch:3d}: loss = {loss:.4f}")

acc = lambda A, t: np.mean(np.argmax(softmax(A @ W + b), axis=1) == t)
print("train accuracy:", round(acc(Xtr, y[tr]), 3), "  test accuracy:", round(acc(Xte, y[te]), 3))

from sklearn.linear_model import LogisticRegression
print("scikit-learn test accuracy:", round(LogisticRegression(max_iter=1000).fit(Xtr, y[tr]).score(Xte, y[te]), 3))

<div dir="rtl" align="right">



## چه چیزی را بررسی کنیم؟
- خطای اولیه باید نزدیک $\ln3=1.099$ باشد (مدل هنوز هیچ نمی‌داند و به هر کلاس احتمال یک‌سوم می‌دهد).
- گرادیان تحلیلی و عددی باید تا چند رقم اعشار یکی باشند.
- خطا باید در هر epoch کم شود؛ اگر زیاد شد نرخ یادگیری بزرگ است.
- دقت نهایی باید به scikit-learn نزدیک باشد.

## ادامه بده
- یک لایه پنهان با ReLU اضافه کن و پس‌انتشار آن را بنویس.
- جریمه L2 به خطا و گرادیان اضافه کن: $\lambda\lVert W\rVert^2$ و $2\lambda W$.
- به جای گرادیان کاهشی ساده، Adam را پیاده کن.
- همین طبقه‌بند را روی دو مؤلفه اصلی PCA آموزش بده و دقت را مقایسه کن.

</div>